# Challenge de la Sesión 09 — Casos de uso: modelos generativos en contexto real
*Use Cases: Generative Models in Real Contexts · Casos de uso: modelos generativos en contexto real*

[![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/09-real-world-use-cases/challenge.ipynb)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/09-real-world-use-cases/challenge.ipynb)

| | |
|---|---|
| **Modalidad** | Individual |
| **Tiempo** | 40 minutos en clase + entrega hasta las **23:59** de hoy (hora de Colombia) |
| **Uso de IA** | Permitido, con **registro obligatorio** al final del notebook |
| **Micro-sustentaciones** | Al azar: si no puedes explicar tu entrega, la nota máxima del challenge es 3.0 |
| **Rúbrica** | Ejecución técnica **40%** · Análisis e interpretación **40%** · Reflexión y registro de IA **20%** ([evaluación del curso](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/evaluacion.md)) |
| **Entrega** | `Archivo → Descargar notebook` y súbelo a e-Aulas como `S09_<codigo>.ipynb` |

**Tu misión.** Tu código estudiantil te asigna un **sector** (salud, finanzas o educación), un **caso** de una organización ficticia y tus **parámetros de ROI**. Vas a medir el prototipo del sector con tu propia muestra, analizar el caso con un canvas, decidir **go / no-go** con tus números y verificar un resumen escrito por IA.

| Tarea | Qué haces | Componente |
|---|---|---|
| 1 | Prototipo del sector y asistente del caso: ejecutar y medir | Ejecución técnica |
| 2 | Canvas del caso apoyado en tus números | Análisis e interpretación |
| 3 | Explica y decide: ROI, payback y decisión go / no-go | Análisis e interpretación |
| 4 | Crítica a la IA: verificar y corregir un resumen ejecutivo | Análisis e interpretación |

**Startti es opcional.** El asistente del caso (Tarea 1b) puede ser un agente de Startti con el documento del caso como base de conocimiento, o la **alternativa en Python** incluida aquí. Las dos vías se califican con la misma rúbrica. Sin clave de API, el notebook usa la alternativa en Python automáticamente.

> ⚠️ Todas las organizaciones y cifras de los casos son **ficticias**. Los prototipos son ejercicios de clase: **no son consejo médico, financiero ni académico**.

**Antes de empezar (Kaggle):** *Settings → Internet* activado. La GPU es opcional (acelera la Tarea 1). Si usas Startti: *Add-ons → Secrets* → `STARTTI_API_KEY`, adjunto a este notebook ([guía de Startti](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-startti.md)).

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S09"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---
ensure("datasets")
ensure("sentence-transformers", "sentence_transformers")
ensure("scikit-learn", "sklearn")

Cliente de Startti del curso (no editar). Si no tienes clave, imprime un aviso y el notebook sigue con la alternativa en Python.

In [ ]:
# --- Startti ADP client ---
import os, requests
STARTTI_BASE_URL = "https://api.startti.ai"

def get_secret(name):
    """Read a secret from Kaggle Secrets, Colab userdata or an environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

STARTTI_API_KEY = get_secret("STARTTI_API_KEY")
_STARTTI_HINTS = {401: "invalid or revoked API key", 402: "plan limit reached",
                  403: "the key cannot access this agent", 404: "agent not found — check AGENT_ID",
                  409: "the agent is not published — publish it in ADP first"}

def startti_run(agent_id, prompt, session_key=None, timeout_s=120):
    """Send one message to a published Startti agent and return its text answer."""
    if not STARTTI_API_KEY:
        raise RuntimeError("STARTTI_API_KEY not found. Add it in Kaggle → Add-ons → Secrets.")
    body = {"agentId": agent_id, "prompt": prompt}
    if session_key:
        body["sessionKey"] = session_key
    r = requests.post(f"{STARTTI_BASE_URL}/v1/run", json=body, timeout=timeout_s,
                      headers={"Authorization": f"Bearer {STARTTI_API_KEY}"})
    if r.status_code >= 400:
        try:
            err = r.json()
        except ValueError:
            err = {"message": r.text[:300]}
        hint = _STARTTI_HINTS.get(r.status_code, "")
        raise RuntimeError(f"Startti API {r.status_code} ({hint}): {err.get('code', '')} {err.get('message', err)}")
    out = r.json().get("output", {})
    if out.get("errorText"):
        raise RuntimeError(f"Agent error: {out['errorText']}")
    if out.get("suspended"):
        questions = "; ".join(q.get("text", "") for q in out.get("suspendQuestions", []))
        return f"[AGENT ASKED] {questions}"
    return out.get("text", "")

print("Startti key found ✅" if STARTTI_API_KEY else "No STARTTI_API_KEY — Startti cells will be skipped.")
# --- end of Startti client ---

In [ ]:
# Tu configuración personal (depende de tu STUDENT_ID: no la cambies)
CASOS = {
    "salud": ["clinica_serena", "telesalud_caribe", "botica_andina"],
    "finanzas": ["andes_bank", "cumbre_capital", "aseguradora_mirador"],
    "educación": ["colegio_los_ceibos", "universidad_colibri", "aula_rural"],
}
# Datos de cada caso, tomados de su documento (data/casos/): volúmenes mensuales posibles,
# minutos que toma hoy cada unidad de trabajo y minutos extra de retrabajo cuando una salida no sirve.
DATOS_CASO = {
    "clinica_serena":      {"volumenes": [4000, 6500, 9000],    "minutos_actuales": 6.0, "minutos_retrabajo": 10.0},
    "telesalud_caribe":    {"volumenes": [6000, 10000, 15000],  "minutos_actuales": 5.0, "minutos_retrabajo": 8.0},
    "botica_andina":       {"volumenes": [20000, 32000, 45000], "minutos_actuales": 4.0, "minutos_retrabajo": 6.0},
    "andes_bank":          {"volumenes": [20000, 40000, 60000], "minutos_actuales": 1.5, "minutos_retrabajo": 2.0},
    "cumbre_capital":      {"volumenes": [15000, 30000, 45000], "minutos_actuales": 1.0, "minutos_retrabajo": 2.0},
    "aseguradora_mirador": {"volumenes": [10000, 20000, 30000], "minutos_actuales": 1.2, "minutos_retrabajo": 2.0},
    "colegio_los_ceibos":  {"volumenes": [1500, 2500, 4000],    "minutos_actuales": 5.0, "minutos_retrabajo": 2.0},
    "universidad_colibri": {"volumenes": [5000, 8000, 12000],   "minutos_actuales": 6.0, "minutos_retrabajo": 3.0},
    "aula_rural":          {"volumenes": [2000, 4000, 6000],    "minutos_actuales": 8.0, "minutos_retrabajo": 4.0},
}
PARAMETRO_SECTOR = {
    "salud": ("condicion_prueba", ["texto completo", "solo la primera frase", "con errores de digitación"]),
    "finanzas": ("verbalizador", ["bearish/bullish/neutral", "negative/positive/neutral", "down/up/flat"]),
    "educación": ("nivel_pregunta", ["recordar", "comprender"]),
}

sector = pick(CASOS)
caso = pick(CASOS[sector])
nombre_parametro, opciones_parametro = PARAMETRO_SECTOR[sector]
CONFIG = {
    "sector": sector,
    "caso": caso,
    nombre_parametro: pick(opciones_parametro),
    "semilla_muestra": pick(range(1000)),
    "volumen_mensual": pick(DATOS_CASO[caso]["volumenes"]),
    "fraccion_ahorro": pick([0.4, 0.6, 0.8]),          # parte del tiempo actual que ahorra una salida utilizable
    "adopcion": pick([0.5, 0.7, 0.9]),                 # parte del volumen en que el equipo realmente usa la herramienta
    "costo_hora_cop": pick([30000, 45000, 60000, 80000]),
    "costo_implementacion_cop": pick([40_000_000, 80_000_000, 150_000_000]),
    "costo_mensual_cop": pick([1_500_000, 3_000_000, 6_000_000]),
}
print("Tu configuración personal")
for k, v in CONFIG.items():
    print(f"  {k}: {v:,}" if isinstance(v, int) and v >= 1000 else f"  {k}: {v}")

## Tu caso

Lee el documento de tu caso con atención: lo necesitas para las cuatro tareas (las cifras de volumen, tiempos y retrabajo salen de aquí). La celda lo busca primero en la carpeta `data/` y, si no está (por ejemplo, en Kaggle), lo descarga del repositorio del curso.

In [ ]:
import re, time, string, collections, unicodedata, urllib.request
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 110)
DATA_URL = "https://raw.githubusercontent.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/main/sessions/09-real-world-use-cases/data/"

def load_text(rel_path):
    """Lee un archivo de datos del curso: primero la copia local y, si no existe, la del repositorio en GitHub."""
    local = Path("data") / rel_path
    if local.exists():
        return local.read_text(encoding="utf-8")
    with urllib.request.urlopen(DATA_URL + rel_path, timeout=30) as resp:
        return resp.read().decode("utf-8")

PREFIJO = {"salud": "salud", "finanzas": "finanzas", "educación": "educacion"}
ARCHIVO_CASO = f"casos/{PREFIJO[CONFIG['sector']]}_{CONFIG['caso']}.md"
documento_caso = load_text(ARCHIVO_CASO)
print(documento_caso)
print(f"Archivo del caso: data/{ARCHIVO_CASO} (para Startti, descárgalo de "
      f"https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/09-real-world-use-cases/data/{ARCHIVO_CASO})")

## Tarea 1 — Prototipo del sector y asistente del caso (Ejecución técnica)

**1a. Prototipo del sector.** Ejecuta el prototipo que corresponde a tu sector sobre **tu muestra personal**:

| Sector | Prototipo | Métrica principal = q |
|---|---|---|
| Finanzas | FinBERT vs. LLM *zero-shot* (con tu verbalizador) en titulares financieros | Exactitud del mejor modelo |
| Salud | Recuperación de casos parecidos con MiniLM (con tu condición de prueba) → 3 condiciones sugeridas | Exactitud top-3 |
| Educación | Generación de preguntas con Qwen2.5-0.5B (con tu nivel) + verificación automática | Tasa de preguntas respondibles |

La celda siguiente define las herramientas (no necesitas editarla, pero sí entenderla: puede salir en la micro-sustentación). Solo se cargan los modelos de tu sector.

In [ ]:
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, recall_score
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

N_MUESTRA = {"finanzas": 16 if FAST_DEV_RUN else 60, "salud": 12 if FAST_DEV_RUN else 40,
             "educación": 2 if FAST_DEV_RUN else 8}
MAX_NEW_TOKENS = 40 if FAST_DEV_RUN else 70
LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
_MODELOS = {}

def get_llm():
    """Carga Qwen2.5-0.5B-Instruct una sola vez."""
    if "llm" not in _MODELOS:
        tok = AutoTokenizer.from_pretrained(LLM_NAME)
        # .float(): las versiones recientes de transformers cargan este modelo en bfloat16, más lento e inestable en CPU
        _MODELOS["llm"] = (tok, AutoModelForCausalLM.from_pretrained(LLM_NAME).float().to(DEVICE).eval())
    return _MODELOS["llm"]

def get_embedder():
    if "emb" not in _MODELOS:
        _MODELOS["emb"] = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    return _MODELOS["emb"]

def chat(messages, max_new_tokens=MAX_NEW_TOKENS, prefix=""):
    """Generación greedy (determinística) con la plantilla de chat. `prefix` inicia la respuesta del modelo."""
    tok, llm = get_llm()
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False) + prefix
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return (prefix + tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)).strip()

# ---------- Finanzas: FinBERT vs. LLM zero-shot ----------
ETIQUETAS_FIN = ["bajista", "alcista", "neutral"]          # ids del dataset: 0 bearish, 1 bullish, 2 neutral
FINBERT_A_DATASET = {"negative": 0, "positive": 1, "neutral": 2}   # por nombre, no por id

def puntajes_etiqueta(texto, verbalizador):
    """Log-probabilidad del primer token de cada palabra-etiqueta como respuesta (una pasada por titular)."""
    tok, llm = get_llm()
    messages = [{"role": "system", "content": "You are a financial news analyst."},
                {"role": "user", "content": "Classify the market sentiment of this financial tweet for an investor.\n\n"
                                            f"Tweet: {texto}\n\nAnswer with one word: {', '.join(verbalizador)}."}]
    enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(DEVICE)
    with torch.no_grad():
        logp = torch.log_softmax(llm(**enc).logits[0, -1].float(), dim=-1)
    puntajes = []
    for palabra in verbalizador:
        ids = sorted({tok(w, add_special_tokens=False)["input_ids"][0] for w in (palabra, palabra.capitalize())})
        puntajes.append(torch.logsumexp(logp[ids], dim=0).item())
    return puntajes

def prototipo_finanzas(config, n):
    datos = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
    muestra = datos.shuffle(seed=config["semilla_muestra"]).select(range(n))
    textos, y = list(muestra["text"]), list(muestra["label"])
    finbert = pipeline("text-classification", model="ProsusAI/finbert", device=DEVICE)
    t0 = time.perf_counter()
    pred_fb = [FINBERT_A_DATASET[o["label"]] for o in finbert(textos, batch_size=16, truncation=True)]
    ms_fb = 1000 * (time.perf_counter() - t0) / n
    verbalizador = config["verbalizador"].split("/")
    t0 = time.perf_counter()
    pred_llm = [int(np.argmax(puntajes_etiqueta(t, verbalizador))) for t in textos]
    ms_llm = 1000 * (time.perf_counter() - t0) / n
    m = {"exactitud_linea_base": collections.Counter(y).most_common(1)[0][1] / n}   # siempre la etiqueta más frecuente
    for nombre, pred, ms in [("finbert", pred_fb, ms_fb), ("llm", pred_llm, ms_llm)]:
        m[f"exactitud_{nombre}"] = accuracy_score(y, pred)
        m[f"macro_f1_{nombre}"] = f1_score(y, pred, average="macro")
        m[f"recall_bajista_{nombre}"] = recall_score(y, pred, labels=[0], average="macro", zero_division=0)
        m[f"ms_por_titular_{nombre}"] = ms
    mejor = "finbert" if m["exactitud_finbert"] >= m["exactitud_llm"] else "llm"
    pred_mejor = pred_fb if mejor == "finbert" else pred_llm
    detalle = pd.DataFrame({"titular": textos, "real": [ETIQUETAS_FIN[i] for i in y],
                            "finbert": [ETIQUETAS_FIN[i] for i in pred_fb], "llm": [ETIQUETAS_FIN[i] for i in pred_llm]})
    errores = detalle[[p != t for p, t in zip(pred_mejor, y)]]
    return {"metricas": m, "q": m[f"exactitud_{mejor}"], "nombre_q": f"exactitud ({mejor})",
            "detalle": detalle, "errores": errores}

# ---------- Salud: recuperación de casos parecidos (no es consejo médico) ----------
def primera_frase(texto):
    return re.split(r"(?<=[.!?])\s+", texto.strip())[0]

def con_errores(texto, tasa=0.15, semilla=42):
    """Borra o intercambia una letra en cerca de `tasa` de las palabras largas (determinístico, como en el lab)."""
    r = random.Random(semilla)
    palabras = []
    for w in texto.split():
        if len(w) > 3 and r.random() < tasa:
            i = r.randrange(1, len(w) - 1)
            w = w[:i] + w[i + 1:] if r.random() < 0.5 else w[:i] + w[i + 1] + w[i] + w[i + 2:]
        palabras.append(w)
    return " ".join(palabras).lower()

TRANSFORMACION = {"texto completo": lambda t: t, "solo la primera frase": primera_frase,
                  "con errores de digitación": con_errores}

def prototipo_salud(config, n, k=5):
    datos = load_dataset("gretelai/symptom_to_diagnosis")
    biblioteca = datos["train"] if not FAST_DEV_RUN else datos["train"].shuffle(seed=0).select(range(300))
    muestra = datos["test"].shuffle(seed=config["semilla_muestra"]).select(range(n))
    emb = get_embedder()
    kb_textos, kb_y = list(biblioteca["input_text"]), list(biblioteca["output_text"])
    kb_vec = emb.encode(kb_textos, normalize_embeddings=True, batch_size=64)
    textos = [TRANSFORMACION[config["condicion_prueba"]](t) for t in muestra["input_text"]]
    y = list(muestra["output_text"])
    sims = emb.encode(textos, normalize_embeddings=True, batch_size=64) @ kb_vec.T
    sugerencias = []
    for fila in sims:
        votos = collections.defaultdict(float)
        for i in np.argsort(-fila)[:k]:
            votos[kb_y[i]] += float(fila[i])
        sugerencias.append([c for c, _ in sorted(votos.items(), key=lambda kv: -kv[1])][:3])
    top1 = float(np.mean([s[0] == t for s, t in zip(sugerencias, y)]))
    top3 = float(np.mean([t in s for s, t in zip(sugerencias, y)]))
    detalle = pd.DataFrame({"descripcion": textos, "real": y, "sugerencias": [", ".join(s) for s in sugerencias]})
    errores = detalle[[t not in s for s, t in zip(sugerencias, y)]]
    return {"metricas": {"exactitud_top1": top1, "exactitud_top3": top3}, "q": top3, "nombre_q": "exactitud top-3",
            "detalle": detalle, "errores": errores}

# ---------- Educación: generación de preguntas + verificación automática ----------
NIVELES = {"recordar": "a factual recall question (who, what, which or where)",
           "comprender": "a question that asks why or how something happens"}
STOPWORDS = set("the a an of to in and or is are was were be by for with on at as it its this that "
                "which what who how why from into than then".split())

def tokens_contenido(texto):
    texto = texto.lower().translate(str.maketrans("", "", string.punctuation))
    return [w for w in texto.split() if w not in STOPWORDS]

def f1_tokens(pred, ref):
    p, r = tokens_contenido(pred), tokens_contenido(ref)
    comunes = sum((collections.Counter(p) & collections.Counter(r)).values())
    if not p or not r or comunes == 0:
        return 0.0
    prec, rec = comunes / len(p), comunes / len(r)
    return 2 * prec * rec / (prec + rec)

def anclada(respuesta, parrafo, minimo=0.75):
    a, par = tokens_contenido(respuesta), set(tokens_contenido(parrafo))
    return bool(a) and sum(w in par for w in a) / len(a) >= minimo

def prototipo_educacion(config, n):
    datos = load_dataset("allenai/sciq", split="validation").filter(lambda r: len(r["support"].split()) >= 40)
    muestra = datos.shuffle(seed=config["semilla_muestra"]).select(range(n))
    filas = []
    for row in muestra:
        par = row["support"]
        crudo = chat([{"role": "system", "content": "You write short quiz questions for high-school science students."},
                      {"role": "user", "content": f"Paragraph:\n{par}\n\nWrite {NIVELES[config['nivel_pregunta']]} that can "
                                                  "be answered using ONLY this paragraph, and its short answer (at most 6 "
                                                  "words).\nUse exactly this format:\nQuestion: ...\nAnswer: ..."}],
                     prefix="Question:")
        q = re.search(r"Question:\s*(.+)", crudo)
        a = re.search(r"Answer:\s*(.+)", crudo)
        q, a = (q.group(1).strip() if q else ""), (a.group(1).strip() if a else "")
        re_resp = chat([{"role": "system", "content": "Answer using only the paragraph, in at most 6 words. If the "
                                                      "paragraph does not contain the answer, reply: not in the paragraph."},
                        {"role": "user", "content": f"Paragraph:\n{par}\n\nQuestion: {q}"}], max_new_tokens=16) if q else ""
        ok_anclada, f1 = anclada(a, par), f1_tokens(re_resp, a)
        filas.append({"pregunta": q, "respuesta": a, "re_respuesta": re_resp, "anclada": ok_anclada,
                      "f1_ida_vuelta": round(f1, 2), "respondible": bool(q) and ok_anclada and f1 >= 0.5})
    detalle = pd.DataFrame(filas)
    m = {"tasa_respondibles": float(detalle["respondible"].mean()), "tasa_ancladas": float(detalle["anclada"].mean()),
         "f1_ida_vuelta_promedio": float(detalle["f1_ida_vuelta"].mean())}
    return {"metricas": m, "q": m["tasa_respondibles"], "nombre_q": "tasa de preguntas respondibles",
            "detalle": detalle, "errores": detalle[~detalle["respondible"]]}

PROTOTIPOS = {"finanzas": prototipo_finanzas, "salud": prototipo_salud, "educación": prototipo_educacion}
print("Herramientas listas.")

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — ejecuta tu prototipo (puedes agregar análisis, pero no cambies la muestra ni el parámetro)
t0 = time.perf_counter()
resultado = PROTOTIPOS[CONFIG["sector"]](CONFIG, N_MUESTRA[CONFIG["sector"]])
print(f"Sector: {CONFIG['sector']} · muestra: {N_MUESTRA[CONFIG['sector']]} · {time.perf_counter() - t0:.0f} s")
for nombre, valor in resultado["metricas"].items():
    print(f"  {nombre}: {valor:.4f}")
print(f"\nq para el ROI = {resultado['nombre_q']} = {resultado['q']:.4f}")
print("\nAlgunos errores del prototipo (revísalos):")
display(resultado["errores"].head(5))

**1b. Asistente del caso.** Un asistente que responde preguntas sobre las reglas de tu caso usando **solo el documento del caso**. Se prueba con 4 preguntas: 3 que el documento responde (la respuesta debe contener el dato correcto) y 1 que **no** responde (lo correcto es decir que no tiene la información o remitir a una persona).

- **Vía A · Startti (opcional):** crea un agente, sube el archivo de tu caso (la ruta y el enlace de descarga aparecen al final del documento, arriba) a una colección de conocimiento, conecta la búsqueda en la base de conocimiento, pega estas instrucciones, **publícalo** y copia su `AGENT_ID` en la celda de abajo.

```text
Eres el asistente virtual de una organización ficticia. Responde en español, en máximo 2 frases,
usando solo la base de conocimiento (el documento del caso).
Si el documento no contiene la respuesta, responde: "No tengo esa información en el documento.
Por favor contacta a una persona del equipo."
Nunca des diagnósticos, recomendaciones de inversión ni datos que no estén en el documento.
```

- **Vía B · Python (sin cuenta):** recuperación híbrida (MiniLM + TF-IDF, porque MiniLM se entrenó sobre todo en inglés) sobre los fragmentos del documento y una respuesta redactada por Qwen2.5-0.5B con el texto recuperado. Si ningún fragmento se parece lo suficiente, responde que no tiene la información.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower()) if unicodedata.category(c) != "Mn")

def fragmentos(markdown):
    """Un fragmento por sección '## '; en las listas, un fragmento por viñeta (con el título de la sección)."""
    salida = []
    for parte in re.split(r"\n(?=## )", markdown):
        parte = parte.strip()
        if not parte.startswith("## "):
            continue
        titulo, _, cuerpo = parte.partition("\n")
        titulo = titulo[3:].strip()
        vinetas = [l[2:].strip() for l in cuerpo.splitlines() if l.startswith("- ")]
        if vinetas:
            salida += [f"{titulo}: {v}" for v in vinetas]
        elif cuerpo.strip():
            salida.append(f"{titulo}. " + re.sub(r"\s+", " ", cuerpo).strip())
    return salida

FRAGMENTOS = fragmentos(documento_caso)
_tfidf = TfidfVectorizer(preprocessor=sin_tildes, ngram_range=(1, 2), sublinear_tf=True).fit(FRAGMENTOS)
_frag_tfidf = _tfidf.transform(FRAGMENTOS)
_frag_emb = get_embedder().encode(FRAGMENTOS, normalize_embeddings=True)
SIMILITUD_MINIMA = 0.25
MODO_ASISTENTE = "rag"          # "rag" (recuperación + LLM) o "recuperacion" (devuelve el fragmento, más barato)
NEGATIVA = "No tengo esa información en el documento. Por favor contacta a una persona del equipo."

def recuperar(pregunta, k=3, alfa=0.5):
    denso = _frag_emb @ get_embedder().encode([pregunta], normalize_embeddings=True)[0]
    lexico = (_frag_tfidf @ _tfidf.transform([pregunta]).T).toarray().ravel()
    puntaje = alfa * denso + (1 - alfa) * lexico
    top = np.argsort(-puntaje)[:k]
    return [FRAGMENTOS[i] for i in top], float(puntaje[top[0]])

def asistente_python(pregunta):
    pasajes, mejor = recuperar(pregunta)
    if mejor < SIMILITUD_MINIMA:
        return NEGATIVA
    if MODO_ASISTENTE == "recuperacion":
        return pasajes[0]
    return chat([{"role": "system", "content": "Eres el asistente virtual de una organización. Responde en español, en "
                                               "máximo 2 frases, usando solo el contexto. Si el contexto no contiene la "
                                               f"respuesta, responde exactamente: {NEGATIVA}"},
                 {"role": "user", "content": "Contexto:\n" + "\n".join(pasajes) + f"\n\nPregunta: {pregunta}"}])

# ✏️ Vía A (opcional): pega el ID de tu agente publicado en Startti
AGENT_ID = ""
USAR_STARTTI = bool(STARTTI_API_KEY) and bool(AGENT_ID.strip())
if USAR_STARTTI:
    try:
        print("Startti:", startti_run(AGENT_ID.strip(), "Hola, ¿qué puedes hacer por mí?"))
    except Exception as err:  # agente sin publicar, ID incorrecto, red…
        USAR_STARTTI = False
        print("La llamada a Startti falló; se usará la alternativa en Python:", err)
VIA_ASISTENTE = "startti" if USAR_STARTTI else f"python-{MODO_ASISTENTE}"
print(f"{len(FRAGMENTOS)} fragmentos del documento · vía del asistente: {VIA_ASISTENTE}")

In [ ]:
# Preguntas de prueba por caso: "espera" = grupos de palabras aceptables (todos los grupos deben aparecer);
# None = pregunta fuera del documento: lo correcto es negarse o remitir a una persona.
PREGUNTAS = {
    "clinica_serena": [
        ("¿Con cuánta anticipación puedo cancelar una cita sin pagar multa?", [["24"]]),
        ("¿En qué horario atienden las citas?", [["7:00", "7 a", "7am", "siete"], ["19"]]),
        ("Tengo dolor en el pecho, ¿qué debo hacer?", [["123"]]),
        ("¿La clínica tiene parqueadero para visitantes?", None)],
    "telesalud_caribe": [
        ("¿En cuánto tiempo me responde el médico de turno?", [["30"]]),
        ("¿En qué horario funciona la línea nocturna?", [["22"], ["6"]]),
        ("¿Quién puede ver mis conversaciones?", [["equipo medico", "medico"]]),
        ("¿Hacen visitas médicas a domicilio en Santa Marta?", None)],
    "botica_andina": [
        ("¿Me pueden vender un antibiótico sin fórmula médica?", [["formula"]]),
        ("¿Hasta qué hora funcionan los domicilios?", [["21"]]),
        ("¿Puedo devolver un medicamento que ya compré?", [["defectuos", "vencid"]]),
        ("¿Tienen vacunación contra la fiebre amarilla?", None)],
    "andes_bank": [
        ("¿En cuánto tiempo se debe revisar una noticia bajista sobre una empresa de un cliente?", [["4"]]),
        ("¿Se puede compartir el resumen matutino con los clientes?", [["interno"]]),
        ("¿A qué hora se envía el resumen a los asesores?", [["7:00", "7 a", "7am", "siete"]]),
        ("¿Cuál es la tasa de interés de los CDT de Andes Bank hoy?", None)],
    "cumbre_capital": [
        ("¿A qué hora se envían las alertas a los clientes?", [["7:30"]]),
        ("¿Quién debe validar una alerta roja?", [["analista"]]),
        ("¿Durante cuánto tiempo se conserva el registro de las alertas?", [["5", "cinco"]]),
        ("¿Cuánto cobran de comisión por comprar acciones en Nueva York?", None)],
    "aseguradora_mirador": [
        ("¿En cuánto tiempo se escala una alerta reputacional?", [["2", "dos"]]),
        ("¿Qué día se reúne el Comité de Inversiones?", [["martes"]]),
        ("¿Cuándo se entrega el informe semanal de riesgos?", [["lunes"]]),
        ("¿Cuánto cuesta el seguro de mi carro?", None)],
    "colegio_los_ceibos": [
        ("¿Cuántas preguntas tiene cada quiz semanal?", [["10", "diez"]]),
        ("¿Qué día se publican los quices?", [["lunes"]]),
        ("¿Quién aprueba las preguntas antes de que las vean los estudiantes?", [["docente", "profesor"]]),
        ("¿Cuánto cuesta la matrícula del colegio?", None)],
    "universidad_colibri": [
        ("¿En cuánto tiempo se revisa un reporte de error de una pregunta?", [["3", "tres"]]),
        ("¿Qué niveles de dificultad tienen las preguntas?", [["basico"], ["intermedio"], ["avanzado"]]),
        ("¿Qué porcentaje de las preguntas generadas revisan los tutores?", [["20"]]),
        ("¿Puedo pagar la matrícula con tarjeta de crédito?", None)],
    "aula_rural": [
        ("¿Qué día se sincronizan las tabletas?", [["viernes"]]),
        ("¿Cuántas tabletas recibe cada escuela?", [["30", "treinta"]]),
        ("¿Cómo deben estar escritas las preguntas?", [["sencillo"]]),
        ("¿Cuál es el salario de un docente rural?", None)],
}
MARCAS_NEGATIVA = ["no tengo", "no cuento", "no dispongo", "no encuentro", "no aparece", "no esta en", "no puedo proporcionar",
                   "no se encuentra", "no hay informacion", "contacta", "comunicate", "una persona del equipo"]

def aprueba(respuesta, espera):
    texto = sin_tildes(respuesta)
    if espera is None:
        return any(m in texto for m in MARCAS_NEGATIVA)
    return all(any(sin_tildes(k) in texto for k in grupo) for grupo in espera)

asistente = (lambda p: startti_run(AGENT_ID.strip(), p)) if USAR_STARTTI else asistente_python
filas = []
for pregunta, espera in PREGUNTAS[CONFIG["caso"]]:
    try:
        respuesta = asistente(pregunta)
    except Exception as err:  # una llamada fallida cuenta como respuesta fallida
        respuesta = f"[error] {err}"
    filas.append({"tipo": "en el documento" if espera else "fuera del documento", "pregunta": pregunta,
                  "respuesta": respuesta.replace("\n", " ")[:160], "aprueba": aprueba(respuesta, espera)})
evaluacion_asistente = pd.DataFrame(filas)
display(evaluacion_asistente)
tasa_asistente = float(evaluacion_asistente["aprueba"].mean())
print(f"Tasa de respuestas correctas del asistente ({VIA_ASISTENTE}): {tasa_asistente:.2f}")

**Interpreta tus resultados (3–5 líneas).** ¿Qué número obtuviste en 1a y qué significa para tu caso? (Finanzas: compáralo también con `exactitud_linea_base`, que siempre responde la etiqueta más frecuente.) Mira los errores que imprimió la celda: ¿qué tienen en común? En 1b, ¿falló alguna pregunta? ¿Fue un error de recuperación, de redacción o de la prueba de palabras clave?

**Tu respuesta:**


## Tarea 2 — Canvas del caso (Análisis e interpretación)

Completa el canvas de **tu** caso. Cada puntaje (1–5) debe justificarse con **evidencia**: un número de la Tarea 1, un dato del documento del caso o una fuente citada en clase. Frases genéricas ("la IA ahorra tiempo") no suman puntos.

**Tu respuesta:**

**Enunciado del caso de uso:** Para ___ que ___, un modelo que ___ produce ___, medido por ___, con ___ (control humano).

| Bloque | Puntaje (1–5) | Justificación con evidencia |
|---|---|---|
| Valor para la organización | | |
| Factibilidad técnica (datos, modelo, idioma) | | |
| Factibilidad organizacional (personas, procesos) | | |
| Riesgo 1 + control | — | |
| Riesgo 2 + control | — | |
| KPI operativo (línea base → meta) | — | |
| KPI de negocio | — | |
| Guardrail (qué vigilarías para detenerlo) | — | |

**Cuadrante en la matriz valor × factibilidad y por qué:**


## Tarea 3 — Explica y decide: ROI y go / no-go (Análisis e interpretación)

La celda calcula el ROI con la misma fórmula de las slides y con **tus** parámetros:

$$H = \frac{V \cdot a \cdot \big(q\,m - (1-q)\,r\big)}{60}, \qquad B = H \cdot c - C_{\text{mensual}}, \qquad \text{payback} = \frac{C_{\text{impl}}}{B}$$

donde $m$ = minutos actuales × fracción de ahorro, y $r$ = minutos de retrabajo del documento. Por defecto usa como $q$ el número que mediste en la Tarea 1a. Puedes cambiar `Q_USADA` **solo si lo justificas** (por ejemplo, si crees que en producción será menor que en tu muestra).

Además calcula un **escenario pesimista** (q − 0.10, adopción − 0.2 y volumen × 0.7).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — si cambias Q_USADA, justifica el cambio en tu respuesta
Q_USADA = resultado["q"]

def calcular_roi(volumen, adopcion, q, minutos_ahorrados, minutos_retrabajo, costo_hora, costo_impl, costo_mensual):
    horas = volumen * adopcion * (q * minutos_ahorrados - (1 - q) * minutos_retrabajo) / 60
    bruto = horas * costo_hora
    neto = bruto - costo_mensual
    costo_12 = costo_impl + 12 * costo_mensual
    return {"horas_netas_mes": horas, "beneficio_bruto_mes_cop": bruto, "beneficio_neto_mes_cop": neto,
            "payback_meses": costo_impl / neto if neto > 0 else None,
            "roi_12_meses": (12 * bruto - costo_12) / costo_12}

datos = DATOS_CASO[CONFIG["caso"]]
m_ahorro = datos["minutos_actuales"] * CONFIG["fraccion_ahorro"]
comun = dict(minutos_ahorrados=m_ahorro, minutos_retrabajo=datos["minutos_retrabajo"], costo_hora=CONFIG["costo_hora_cop"],
             costo_impl=CONFIG["costo_implementacion_cop"], costo_mensual=CONFIG["costo_mensual_cop"])
roi_base = calcular_roi(CONFIG["volumen_mensual"], CONFIG["adopcion"], Q_USADA, **comun)
roi_pesimista = calcular_roi(0.7 * CONFIG["volumen_mensual"], max(0.1, CONFIG["adopcion"] - 0.2),
                             max(0.0, Q_USADA - 0.10), **comun)
q_equilibrio = datos["minutos_retrabajo"] / (m_ahorro + datos["minutos_retrabajo"])

tabla_roi = pd.DataFrame({"base": roi_base, "pesimista": roi_pesimista}).T
print(f"q usada = {Q_USADA:.4f} · minutos ahorrados por salida útil m = {m_ahorro:.2f} · retrabajo r = {datos['minutos_retrabajo']}")
print(f"Con q por debajo de {q_equilibrio:.2f}, el retrabajo se come todo el ahorro de tiempo.")
display(tabla_roi.style.format({"horas_netas_mes": "{:,.0f}", "beneficio_bruto_mes_cop": "{:,.0f}",
                                "beneficio_neto_mes_cop": "{:,.0f}", "payback_meses": "{:,.1f}",
                                "roi_12_meses": "{:.1%}"}, na_rep="nunca"))

# ✏️ Escribe tu decisión: "go", "go condicionado" o "no-go"
DECISION = ""

**Explica y decide (5–8 líneas).** ¿Go, go condicionado o no-go? Justifícalo con **tus** números: payback y ROI en los dos escenarios, el q que mediste y el q de equilibrio, y el riesgo principal de tu canvas. Si es go condicionado, di qué condición medible debe cumplirse y cuál sería tu criterio para detener el proyecto (*kill criterion*). Indica qué supuesto de la tabla es el más débil y cómo lo verificarías en un piloto.

**Tu respuesta:**


## Tarea 4 — Crítica a la IA (Análisis e interpretación)

Un colega le pidió a un asistente de IA un **resumen ejecutivo** de tu caso para presentarlo al comité. Este es el texto que obtuvo. Contiene **errores**: afirmaciones que las fuentes no respaldan, una interpretación legal equivocada y datos que no coinciden con lo que tú mediste. También hay afirmaciones correctas: no las marques como errores sin verificarlas.

Para cada error: **cítalo, explica por qué es incorrecto con una fuente verificable** (el artículo, el fallo o tus propios resultados) y **escribe la versión corregida**. Usa las referencias de las slides; puedes consultar los documentos originales.

In [ ]:
ORGANIZACION = documento_caso.splitlines()[0].lstrip("# ").split(" — ")[0]
q_inflada = min(round(resultado["q"] + 0.15, 2), max(round(resultado["q"], 2), 0.95))
AFIRMACION_SECTOR = {
    "salud": "Según Singhal et al. (2023), publicado en Nature, los asistentes de IA generativa reducen en un 60% "
             "los errores de triaje en clínicas de América Latina.",
    "finanzas": "McKinsey (2023) calcula que la IA generativa aumenta en un 45% la rentabilidad de los bancos "
                "latinoamericanos que la adoptan.",
    "educación": "Bastani et al. (2025) demostraron que los estudiantes que usan ChatGPT sin restricciones obtienen "
                 "notas un 48% más altas en los exámenes finales.",
}
RESUMEN_IA = f"""RESUMEN EJECUTIVO — {ORGANIZACION}

1. La IA generativa tiene un potencial económico enorme: McKinsey (2023) estima entre 2,6 y 4,4 billones de
   dólares anuales en los 63 casos de uso que analizó.
2. {AFIRMACION_SECTOR[CONFIG['sector']]}
3. En atención al cliente, Brynjolfsson, Li y Raymond (2023) encontraron que un asistente de IA aumentó en
   promedio un 14% los casos resueltos por hora, con mayores mejoras en los agentes con menos experiencia.
4. Nuestro prototipo se probó con 1.200 casos reales de {ORGANIZACION} y alcanzó una {resultado['nombre_q']} de
   {q_inflada:.0%}, por lo que puede operar sin supervisión humana desde el primer mes.
5. El riesgo legal es bajo: como estableció el caso Moffatt v. Air Canada (2024), una empresa no responde por la
   información incorrecta que da su chatbot si publica la política correcta en otra página de su sitio web,
   porque el chatbot es una entidad independiente.
"""
print(RESUMEN_IA)

**Tu respuesta:**

| # | Afirmación (cita corta) | ¿Correcta? | Evidencia o fuente verificable | Versión corregida |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

**¿Qué te enseña este ejercicio sobre usar IA para escribir casos de negocio? (2–3 líneas)**


In [ ]:
def _r(x):
    return None if x is None else round(float(x), 4)

RESULTS = {
    "sector": CONFIG["sector"],
    "caso": CONFIG["caso"],
    "metricas_prototipo": {k: _r(v) for k, v in resultado["metricas"].items()},
    "q_medida": _r(resultado["q"]),
    "q_usada": _r(Q_USADA),
    "via_asistente": VIA_ASISTENTE,
    "tasa_asistente": _r(tasa_asistente),
    "roi_base": {k: _r(v) for k, v in roi_base.items()},
    "roi_pesimista": {k: _r(v) for k, v in roi_pesimista.items()},
    "q_equilibrio": _r(q_equilibrio),
    "decision": DECISION.strip().lower(),
}
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))